In [ ]:
# Colab Setup: Run this cell first!
%pip install -q openai pydantic

from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 04 - Structured Output

## Why Code Instead of ChatGPT? Reason #3: CONTROL

When you ask ChatGPT to analyze a text, you get back **paragraphs**. That's fine for reading, but what if you want to:
- Compare the same fields across 100 texts?
- Build a database of literary features?
- Generate player cards or encyclopedia entries?

You can't easily do that with paragraphs. You need **structured data**—organized information with consistent fields.

The API gives you access to **structured outputs**: you tell the AI exactly what format you want, and it gives you back clean, organized data.

**Once you have structured data, you can build things from it.**

---

## Setup

In [ ]:
import os

if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY not found! Add it to Colab Secrets.")

print("✓ API key found")

from openai import OpenAI
client = OpenAI()
print("✓ OpenAI client ready")

In [ ]:
# A passage to analyze
passage = """In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters. And God said, Let there be light: and there was light. And God saw the light, that it was good: and God divided the light from the darkness. And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day."""

---

## The Problem with Paragraphs

Let's ask for analysis the normal way:

In [ ]:
def ask(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Normal analysis
result = ask(f"""Analyze this passage. Identify the title/source, main themes, a representative quote, and the overall mood.

{passage}""")

print(result)

That's a nice paragraph, but:
- How do you extract just the themes?
- How do you compare this to 100 other passages?
- How do you build a database or a card from this?

You'd have to parse the text manually. That's fragile and error-prone.

---

## Step 1: Vibecode Structured Output

Go to ChatGPT/Claude/Gemini and type:

> **Write Python code that uses the OpenAI API with Pydantic to get structured output. I want to analyze a text passage and get back a structured object with these fields: `title` (string), `themes` (list of strings), `quote` (string), `mood` (string). Use `client.beta.chat.completions.parse()` with a Pydantic model. I already have `client = OpenAI()` set up.**

In [ ]:
# Here's what the AI might give you:

from pydantic import BaseModel

class TextAnalysis(BaseModel):
    title: str
    themes: list[str]
    quote: str
    mood: str

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": f"""Analyze this passage and extract the requested information:

{passage}"""
        }
    ],
    response_format=TextAnalysis
)

analysis = response.choices[0].message.parsed
print(analysis)

---

## Step 2: Unpack It

Ask the AI:

> **Explain this code to me. What is Pydantic? What is a schema? What does `response_format` do?**

### Understanding the Code

```python
from pydantic import BaseModel

class TextAnalysis(BaseModel):
    title: str
    themes: list[str]
    quote: str
    mood: str
```

- **Pydantic** — A Python library for defining data structures
- **BaseModel** — The base class for defining a schema
- **Schema** — A template that says "I want these fields with these types"
- **Type hints** — `str` means text, `list[str]` means a list of text items

```python
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[...],
    response_format=TextAnalysis
)
```

- **`response_format=TextAnalysis`** — Tells the API: "Give me data that matches this schema"
- The API is *forced* to return structured data, not free-form text

```python
analysis = response.choices[0].message.parsed
```

- **`.parsed`** — Gets the structured data as a Python object
- Now `analysis.title`, `analysis.themes`, etc. are directly accessible

---

## The Key Concepts

| Concept | What it means |
|---------|---------------|
| **Schema** | A template that defines what fields you want and their types |
| **Pydantic** | A Python library for defining schemas as classes |
| **Structured output** | The AI returns data matching your schema, not free-form text |
| **JSON** | A standard format for structured data: `{"key": "value"}` |

The pattern:

```python
from pydantic import BaseModel

class MySchema(BaseModel):
    field1: str
    field2: list[str]
    field3: int

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[...],
    response_format=MySchema
)

data = response.choices[0].message.parsed
```

---

## Step 3: Access the Fields

The beautiful thing about structured output: you can access each field directly.

In [ ]:
# Access individual fields
print("Title:", analysis.title)
print()
print("Themes:", analysis.themes)
print()
print("Quote:", analysis.quote)
print()
print("Mood:", analysis.mood)

In [ ]:
# You can also loop through list fields
print("Themes in this passage:")
for theme in analysis.themes:
    print(f"  - {theme}")

In [ ]:
# Convert to a dictionary if needed
analysis.model_dump()

---

## Step 4: Extend the Schema

**Challenge:** Modify the schema to add a new field called `key_images` that's a list of strings (imagery from the passage).

In [ ]:
# Extended schema with key_images:

class TextAnalysisExtended(BaseModel):
    title: str
    themes: list[str]
    quote: str
    mood: str
    key_images: list[str]  # NEW FIELD

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": f"""Analyze this passage and extract the requested information:

{passage}"""
        }
    ],
    response_format=TextAnalysisExtended
)

extended = response.choices[0].message.parsed
print("Key images:", extended.key_images)

---

## Why Structured Data Matters

With structured data, you can:

1. **Compare consistently** — Every passage has the same fields
2. **Build databases** — Store results in a spreadsheet or database
3. **Create visualizations** — Make charts from the data
4. **Generate artifacts** — Build encyclopedia entries, player cards, etc.

The AI doesn't just *tell* you about the text—it gives you *data* you can use.

---

## Preview: From Data to Cards

In the next notebook, we'll take structured data like this:

```python
{
    "title": "Genesis 1:1-5",
    "themes": ["creation", "light vs darkness", "divine speech"],
    "quote": "Let there be light: and there was light.",
    "mood": "majestic and foundational",
    "key_images": ["void", "deep", "waters", "light", "darkness"]
}
```

And turn it into a **player card** or **encyclopedia entry**.

---

## What You Learned

- **Structured output** gives you organized data, not paragraphs
- **Pydantic schemas** define what fields you want
- You can **access fields directly** from the result
- Structured data lets you **build things** — databases, comparisons, cards

**Next up:** We'll put it all together—loop through 3 passages, extract structured data from each, and generate player cards.